In [1]:
import json
import pandas as pd
import numpy as np
import tqdm as tqdm

import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [2]:
cities = ['regensburg','landshut','bayreuth','schweinfurt','wuerzburg','bamberg']
seed = 42
test_count = 100
seed_idxs = range(1,6)

train_val_configs = ((10, 3),
                     (20, 5),
                     (40, 10),
                     (80, 20),
                     (160, 40))

city_mapping = {
    'regensburg': 'C1',
    'landshut': 'C2',
    'bayreuth': 'C3',
    'schweinfurt': 'C4',
    'wuerzburg': 'C5',
    'bamberg': 'C6'
}

results_dir = '../../data/inductive_gnn_data_results/transductive/Scratch_vs_Finetune/'

metrics = ['top_1_hit_rate', 'top_5_hit_rate', 'top_10_hit_rate', 'bottom_1_hit_rate', 'bottom_5_hit_rate', 'bottom_10_hit_rate',
           'loss', 'r2', 'spearman', 'pearson', 'epochs', 'time']

combined_results = {city: {'scratch': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs},
                          'finetune': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs}} for city in cities}

random_results = {city: {'scratch': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs},
                         'finetune': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs}} for city in cities}

In [3]:
# WandB CSV

efficiency_data = pd.read_csv('efficiency.csv')

for row in efficiency_data.itertuples():
    run_name = row.Name

    if "pretrain" in run_name or "INCOMPLETE" in run_name:
        continue
    
    city = run_name.split('_')[0]
    approach = run_name.split('_')[1]
    train_count = int(run_name.split('_')[-2].split('t')[-1])
    val_count = int(run_name.split('_')[-1].split('v')[-1])

    if (train_count, val_count) not in train_val_configs:
        continue

    combined_results[city][approach][f"train_{train_count}_val_{val_count}"]['epochs'].append(row.epoch+1)
    combined_results[city][approach][f"train_{train_count}_val_{val_count}"]['time'].append(row.Runtime/60)  # Convert to minutes

FileNotFoundError: [Errno 2] No such file or directory: 'efficiency.csv'

In [4]:
for city in cities:
    for train_count, val_count in train_val_configs:
        for seed_idx in seed_idxs:
            
            scratch_run_name = f"{city}_scratch_rs_{seed_idx}_t{train_count}_v{val_count}"
            finetune_run_name = f"{city}_finetune_rs_{seed_idx}_t{train_count}_v{val_count}"
            results_json_name = f"{city}_rs{seed_idx}_t{train_count}_v{val_count}_seed{seed+seed_idx-1}_train{train_count}_val{val_count}_test{test_count}_distant_iou_metrics.json"
            random_results_json_name = f"{city}_rs{seed_idx}_t{train_count}_v{val_count}_seed{seed+seed_idx-1}_train{train_count}_val{val_count}_test{test_count}_random_metrics.json"

            with open(results_dir + scratch_run_name + '/evaluation/' + results_json_name, 'r') as f:
                scratch_results = json.load(f)
            
            with open(results_dir + finetune_run_name + '/evaluation/' + results_json_name, 'r') as f:
                finetune_results = json.load(f)

            with open(results_dir + scratch_run_name + '/evaluation/' + random_results_json_name, 'r') as f:
                scratch_random_results = json.load(f)

            with open(results_dir + finetune_run_name + '/evaluation/' + random_results_json_name, 'r') as f:
                finetune_random_results = json.load(f)

            for metric in metrics:

                if metric in ['epochs', 'time']:
                    continue  # Already recorded from WandB CSV
                
                combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric].append(scratch_results['hit_rates'][metric] if 'hit_rate' in metric else scratch_results[metric])
                combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric].append(finetune_results['hit_rates'][metric] if 'hit_rate' in metric else finetune_results[metric])

                random_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric].append(scratch_random_results['hit_rates'][metric] if 'hit_rate' in metric else scratch_random_results[metric])
                random_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric].append(finetune_random_results['hit_rates'][metric] if 'hit_rate' in metric else finetune_random_results[metric])

### TABLES!

In [5]:
def calc_diff(scratch_x, finetune_x,
              absolute_diff=True, metric_to_perc=False):

      scratch_arr = np.array(scratch_x)*100 if metric_to_perc else np.array(scratch_x)
      finetune_arr = np.array(finetune_x)*100 if metric_to_perc else np.array(finetune_x)

      # diff = finetune_arr - scratch_arr
      # if not absolute_diff:
      #       diff = (diff / scratch_arr) * 100
      
      # return f"{np.mean(diff):.2f} $\\pm$ {np.std(diff):.2f}"

      diff = np.mean(finetune_arr) - np.mean(scratch_arr)
      if not absolute_diff:
            diff = (diff / np.mean(scratch_arr)) * 100

      return f"{diff:.2f}"

In [6]:
# Results Table
train_count, val_count = train_val_configs[2]

for city in cities:
    
    print(f"\\multirow{{4}}{{*}}{{{city_mapping[city]}}}")
    
    scratch_results = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"]
    finetune_results = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"]

    print(f"& Scratch & "
          f"{np.mean(scratch_results['loss']):.2f} $\\pm$ {np.std(scratch_results['loss']):.2f} & "
          f"{np.mean(scratch_results['top_1_hit_rate'])*100:.2f} $\\pm$ {np.std(scratch_results['top_1_hit_rate'])*100:.2f} & "
          f"{np.mean(scratch_results['bottom_1_hit_rate'])*100:.2f} $\\pm$ {np.std(scratch_results['bottom_1_hit_rate'])*100:.2f} & "
          f"{np.mean(scratch_results['r2']):.2f} $\\pm$ {np.std(scratch_results['r2']):.2f} & "
          f"{np.mean(scratch_results['spearman']):.2f} $\\pm$ {np.std(scratch_results['spearman']):.2f} & "
          f"{np.mean(scratch_results['pearson']):.2f} $\\pm$ {np.std(scratch_results['pearson']):.2f} \\\\")
    
    print(f"& Finetune & "
          f"{np.mean(finetune_results['loss']):.2f} $\\pm$ {np.std(finetune_results['loss']):.2f} & "
          f"{np.mean(finetune_results['top_1_hit_rate'])*100:.2f} $\\pm$ {np.std(finetune_results['top_1_hit_rate'])*100:.2f} & "
          f"{np.mean(finetune_results['bottom_1_hit_rate'])*100:.2f} $\\pm$ {np.std(finetune_results['bottom_1_hit_rate'])*100:.2f} & "
          f"{np.mean(finetune_results['r2']):.2f} $\\pm$ {np.std(finetune_results['r2']):.2f} & "
          f"{np.mean(finetune_results['spearman']):.2f} $\\pm$ {np.std(finetune_results['spearman']):.2f} & "
          f"{np.mean(finetune_results['pearson']):.2f} $\\pm$ {np.std(finetune_results['pearson']):.2f} \\\\")
    
    print(f"& Absolute $\\Delta$ & "
          f"{calc_diff(scratch_results['loss'], finetune_results['loss'])} & "
          f"{calc_diff(scratch_results['top_1_hit_rate'], finetune_results['top_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_1_hit_rate'], finetune_results['bottom_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['r2'], finetune_results['r2'])} & "
          f"{calc_diff(scratch_results['spearman'], finetune_results['spearman'])} & "
          f"{calc_diff(scratch_results['pearson'], finetune_results['pearson'])} \\\\")
    
    print(f"& Relative $\\Delta$ & "
          f"{calc_diff(scratch_results['loss'], finetune_results['loss'], absolute_diff=False)} & "
          f"{calc_diff(scratch_results['top_1_hit_rate'], finetune_results['top_1_hit_rate'], absolute_diff=False, metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_1_hit_rate'], finetune_results['bottom_1_hit_rate'], absolute_diff=False, metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['r2'], finetune_results['r2'], absolute_diff=False)} & "
          f"{calc_diff(scratch_results['spearman'], finetune_results['spearman'], absolute_diff=False)} & "
          f"{calc_diff(scratch_results['pearson'], finetune_results['pearson'], absolute_diff=False)} \\\\")
    
    print("\\midrule")

\multirow{4}{*}{C1}
& Scratch & 118.55 $\pm$ 15.79 & 34.49 $\pm$ 4.00 & 40.13 $\pm$ 4.69 & 0.21 $\pm$ 0.11 & 0.29 $\pm$ 0.01 & 0.53 $\pm$ 0.05 \\
& Finetune & 77.75 $\pm$ 11.43 & 46.35 $\pm$ 6.77 & 51.46 $\pm$ 4.25 & 0.47 $\pm$ 0.07 & 0.42 $\pm$ 0.02 & 0.69 $\pm$ 0.05 \\
& Absolute $\Delta$ & -40.80 & 11.87 & 11.33 & 0.27 & 0.13 & 0.16 \\
& Relative $\Delta$ & -34.41 & 34.40 & 28.23 & 129.64 & 45.39 & 30.41 \\
\midrule
\multirow{4}{*}{C2}
& Scratch & 303.12 $\pm$ 132.07 & 60.32 $\pm$ 15.06 & 60.52 $\pm$ 8.35 & 0.62 $\pm$ 0.19 & 0.31 $\pm$ 0.03 & 0.79 $\pm$ 0.12 \\
& Finetune & 115.98 $\pm$ 46.22 & 81.67 $\pm$ 4.15 & 78.40 $\pm$ 3.89 & 0.86 $\pm$ 0.05 & 0.51 $\pm$ 0.02 & 0.93 $\pm$ 0.02 \\
& Absolute $\Delta$ & -187.14 & 21.35 & 17.88 & 0.24 & 0.19 & 0.14 \\
& Relative $\Delta$ & -61.74 & 35.39 & 29.54 & 37.75 & 60.86 & 18.10 \\
\midrule
\multirow{4}{*}{C3}
& Scratch & 24.20 $\pm$ 3.44 & 42.89 $\pm$ 2.17 & 49.81 $\pm$ 3.16 & 0.44 $\pm$ 0.06 & 0.53 $\pm$ 0.02 & 0.66 $\pm$ 0.04 \\
& Finet

In [7]:
# Bring back later if needed
print(f"{np.mean(scratch_results['epochs']):.2f} $\\pm$ {np.std(scratch_results['epochs']):.2f} & "
      f"{np.mean(scratch_results['time']):.2f} $\\pm$ {np.std(scratch_results['time']):.2f} & ")

nan $\pm$ nan & nan $\pm$ nan & 


/opt/anaconda3/envs/chenhao-gnn/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/envs/chenhao-gnn/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/envs/chenhao-gnn/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/chenhao-gnn/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/envs/chenhao-gnn/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [8]:
# Performance Difference: Distant vs Random Test Sets
train_count, val_count = train_val_configs[2]

for city in cities:
    print("\\midrule")
    
    scratch_results = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"]
    finetune_results = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"]
    random_scratch_results = random_results[city]['scratch'][f"train_{train_count}_val_{val_count}"]
    random_finetune_results = random_results[city]['finetune'][f"train_{train_count}_val_{val_count}"]

    print(f"{city_mapping[city]} & Scratch & "
          f"{calc_diff(scratch_results['top_1_hit_rate'], random_scratch_results['top_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_1_hit_rate'], random_scratch_results['bottom_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['top_5_hit_rate'], random_scratch_results['top_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_5_hit_rate'], random_scratch_results['bottom_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['top_10_hit_rate'], random_scratch_results['top_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_10_hit_rate'], random_scratch_results['bottom_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['loss'], random_scratch_results['loss'])} & "
          f"{calc_diff(scratch_results['r2'], random_scratch_results['r2'])} & "
          f"{calc_diff(scratch_results['spearman'], random_scratch_results['spearman'])} & "
          f"{calc_diff(scratch_results['pearson'], random_scratch_results['pearson'])} \\\\")
    
    print(f"{city_mapping[city]} & Finetune & "
          f"{calc_diff(finetune_results['top_1_hit_rate'], random_finetune_results['top_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['bottom_1_hit_rate'], random_finetune_results['bottom_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['top_5_hit_rate'], random_finetune_results['top_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['bottom_5_hit_rate'], random_finetune_results['bottom_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['top_10_hit_rate'], random_finetune_results['top_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['bottom_10_hit_rate'], random_finetune_results['bottom_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['loss'], random_finetune_results['loss'])} & "
          f"{calc_diff(finetune_results['r2'], random_finetune_results['r2'])} & "
          f"{calc_diff(finetune_results['spearman'], random_finetune_results['spearman'])} & "
          f"{calc_diff(finetune_results['pearson'], random_finetune_results['pearson'])} \\\\")

\midrule
C1 & Scratch & 23.42 & 34.00 & 20.41 & 22.35 & 16.07 & 12.95 & 22.43 & 0.47 & 0.14 & 0.30 \\
C1 & Finetune & 26.26 & 37.23 & 27.55 & 25.46 & 20.57 & 16.16 & -26.29 & 0.41 & 0.17 & 0.25 \\
\midrule
C2 & Scratch & 29.80 & 26.21 & 24.59 & 21.75 & 21.50 & 12.69 & -178.51 & 0.30 & 0.17 & 0.17 \\
C2 & Finetune & 12.33 & 14.46 & 17.49 & 20.98 & 18.38 & 12.76 & -66.22 & 0.11 & 0.14 & 0.05 \\
\midrule
C3 & Scratch & 24.38 & 28.89 & 6.17 & 12.38 & 4.82 & 6.64 & -0.94 & 0.40 & 0.05 & 0.25 \\
C3 & Finetune & 27.16 & 32.32 & 10.24 & 16.98 & 5.61 & 8.56 & -3.84 & 0.37 & 0.05 & 0.22 \\
\midrule
C4 & Scratch & 18.31 & 34.13 & 12.95 & 14.39 & 9.44 & 7.93 & 3.36 & 0.26 & 0.07 & 0.15 \\
C4 & Finetune & 10.32 & 24.48 & 14.75 & 11.66 & 9.56 & 6.66 & 0.22 & 0.15 & 0.06 & 0.08 \\
\midrule
C5 & Scratch & 47.61 & 40.19 & 16.30 & 13.82 & 10.07 & 7.00 & 10.19 & 0.54 & 0.07 & 0.36 \\
C5 & Finetune & 49.86 & 41.52 & 15.60 & 13.38 & 8.80 & 6.84 & -11.05 & 0.52 & 0.06 & 0.34 \\
\midrule
C6 & Scratch & 28.20

In [9]:
# Hit Rates Table
train_count, val_count = train_val_configs[2]

for direction in ["top", "bottom"]:
    for k in [1, 5, 10]:
        metric = f"{direction}_{k}_hit_rate"
        print("\\midrule")
        for city in cities:
            scratch_values = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric]
            finetune_values = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric]
            diff_values = np.array(finetune_values) - np.array(scratch_values)

            print(f"{direction.capitalize()} & {k} & {city_mapping[city]} & "
                  f"{np.mean(scratch_values)*100:.2f} $\\pm$ {np.std(scratch_values)*100:.2f} & "
                  f"{np.mean(finetune_values)*100:.2f} $\\pm$ {np.std(finetune_values)*100:.2f} & "
                  f"{np.mean(diff_values)*100:.2f} $\\pm$ {np.std(diff_values)*100:.2f} \\\\")

\midrule
Top & 1 & C1 & 34.49 $\pm$ 4.00 & 46.35 $\pm$ 6.77 & 11.87 $\pm$ 6.93 \\
Top & 1 & C2 & 60.32 $\pm$ 15.06 & 81.67 $\pm$ 4.15 & 21.35 $\pm$ 16.72 \\
Top & 1 & C3 & 42.89 $\pm$ 2.17 & 46.90 $\pm$ 2.52 & 4.01 $\pm$ 2.27 \\
Top & 1 & C4 & 46.04 $\pm$ 6.39 & 68.22 $\pm$ 2.34 & 22.18 $\pm$ 4.38 \\
Top & 1 & C5 & 25.64 $\pm$ 1.86 & 29.36 $\pm$ 1.40 & 3.73 $\pm$ 2.46 \\
Top & 1 & C6 & 31.66 $\pm$ 7.72 & 49.87 $\pm$ 4.92 & 18.21 $\pm$ 5.57 \\
\midrule
Top & 5 & C1 & 42.90 $\pm$ 1.47 & 49.59 $\pm$ 2.25 & 6.69 $\pm$ 2.26 \\
Top & 5 & C2 & 54.99 $\pm$ 8.90 & 69.41 $\pm$ 4.83 & 14.42 $\pm$ 7.82 \\
Top & 5 & C3 & 52.63 $\pm$ 0.95 & 55.80 $\pm$ 1.78 & 3.16 $\pm$ 1.27 \\
Top & 5 & C4 & 43.51 $\pm$ 2.61 & 55.78 $\pm$ 1.23 & 12.27 $\pm$ 1.49 \\
Top & 5 & C5 & 35.43 $\pm$ 1.95 & 41.83 $\pm$ 0.60 & 6.41 $\pm$ 1.76 \\
Top & 5 & C6 & 39.32 $\pm$ 2.64 & 47.62 $\pm$ 2.24 & 8.31 $\pm$ 1.51 \\
\midrule
Top & 10 & C1 & 45.87 $\pm$ 0.59 & 51.75 $\pm$ 1.31 & 5.88 $\pm$ 1.31 \\
Top & 10 & C2 & 53.30 $\pm$ 

### Test Distances

In [10]:
def compute_test_distances(train_count, val_count, test_set_type="distant_iou"):
    
    test_distances = {city: [] for city in cities}

    for city in cities:
        for seed_idx in seed_idxs:
            
            test_split_json_path = f'../../data/splits/{city}/rs_{seed_idx}/t{train_count}_v{val_count}/'
            test_split_json_path += f'{city}_rs{seed_idx}_t{train_count}_v{val_count}_seed{42+seed_idx-1}_train{train_count}_val{val_count}_test{test_count}_{test_set_type}.json'

            with open(test_split_json_path, 'r') as f:
                test_split = json.load(f)

            # Multiple ways here!
            # test_distances_when_picked = np.array(test_split["test_distances_when_picked"])
            
            test_distances_from_train = np.array(test_split["test_distances_from_train"])
            test_distances_from_val = np.array(test_split["test_distances_from_val"])
            
            # test_distances_combi = np.minimum(test_distances_from_train, test_distances_from_val)
            test_distances_combi = 0.8 * test_distances_from_train + 0.2 * test_distances_from_val

            test_distances[city].extend(test_distances_combi.tolist())

    return test_distances

In [11]:
iou_test_distances = compute_test_distances(train_count=40, val_count=10, test_set_type="distant_iou")
random_test_distances = compute_test_distances(train_count=40, val_count=10, test_set_type="random")

In [12]:
print("IOU Test Distances:")
for city in cities:
    distances = iou_test_distances[city]
    print(f"{city.capitalize()}: {np.mean(distances):.6f} ± {np.std(distances):.6f}")

print("\nRandom Test Distances:")
for city in cities:
    distances = random_test_distances[city]
    print(f"{city.capitalize()}: {np.mean(distances):.6f} ± {np.std(distances):.6f}")

IOU Test Distances:
Regensburg: 0.932676 ± 0.027673
Landshut: 0.923782 ± 0.035354
Bayreuth: 0.930366 ± 0.035837
Schweinfurt: 0.870353 ± 0.038264
Wuerzburg: 0.942378 ± 0.028312
Bamberg: 0.886649 ± 0.038412

Random Test Distances:
Regensburg: 0.817969 ± 0.047787
Landshut: 0.826980 ± 0.054656
Bayreuth: 0.823452 ± 0.052659
Schweinfurt: 0.790763 ± 0.056689
Wuerzburg: 0.824766 ± 0.052043
Bamberg: 0.782838 ± 0.057472


### Correlation plots (outdated!)

In [13]:
color_map = {
    'Scratch: Top 1': 'blue',
    'Finetune: Top 1': 'green',
    'Scratch: Top 5': 'red',
    'Finetune: Top 5': 'orange',}

In [14]:
def plot_sample_efficiency(metrics, city):

    fig = plt.figure(figsize=(10, 6))

    for metric in metrics:
        
        scratch_points = []
        finetune_points = []

        for train_count, val_count in train_val_configs:
            
            scratch_values = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric]
            finetune_values = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric]
            
            scratch_points.extend([(train_count + val_count), val] for val in scratch_values)
            finetune_points.extend([(train_count + val_count), val] for val in finetune_values)

        scratch_label = 'Scratch: ' + metric.split('_hit_rate')[0].replace('_', ' ').capitalize()
        finetune_label = 'Finetune: ' + metric.split('_hit_rate')[0].replace('_', ' ').capitalize()

        for points, label, color in [(scratch_points, scratch_label, color_map[scratch_label]),
                                     (finetune_points, finetune_label, color_map[finetune_label])]:
            x, y = zip(*points)
            plt.scatter(x, y, label=label, alpha=0.7, color=color)

        # Fit and plot regression lines
        for points, color in [(scratch_points, color_map[scratch_label]),
                              (finetune_points, color_map[finetune_label])]:
            x, y = zip(*points)
            x = np.array(x).reshape(-1, 1)
            y = np.array(y)
            model = LinearRegression()
            model.fit(x, y)
            x_range = np.linspace(min(x), max(x), 100).reshape(-1, 1)
            y_pred = model.predict(x_range)
            plt.plot(x_range, y_pred, color=color, linestyle='--', alpha=0.7)

    plt.xlabel('Number of Samples')
    # plt.ylabel(metric.replace('_', ' ').capitalize())
    plt.ylabel('Hit Rate')
    plt.title(f'Sample efficiency for {city.capitalize()}')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    # plt.savefig(f'plots/sample_efficiency/{city}_{metric}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'plots/sample_efficiency/{city}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [15]:
for direction in ["top", "bottom"]:
    for k in [1, 5, 10]:
        metric = f"{direction}_{k}_hit_rate"
        for city in cities:
            plot_sample_efficiency(metric, city)

KeyError: 't'

<Figure size 1000x600 with 0 Axes>

In [ ]:
plot_sample_efficiency(['top_1_hit_rate', 'top_5_hit_rate'], 'schweinfurt')